# Subalgebras, ideals and quotients

A vertex algebra $V$ admits notions of subalgebras and quotients, and this notebook shows how to construct and compute these. Specifically, the notebook covers:

* **subalgebras**: `V.subalgebra(A)` is the vertex subalgebra generated by a list $A$ of elements of $V$. The library finds a set of *strong generators* for it: elements whose normally ordered monomials ${:}\partial^{k_1}g_1 \cdots \partial^{k_r}g_r{:}$ span the subalgebra [Kac1998, DSK2005];
* **ideals**: `V.ideal(A)` is the ideal generated by $A$, the smallest subspace containing $A$ that is stable under $\partial$ and under all products $a_{(n)}$, $a \in V$, $n \in \mathbb Z$. It comes with exact membership tests, certificates of membership and normal forms [KK2023];
* **quotients**: `V.quotient(I)`, or `V / I`, is the quotient vertex algebra by an ideal $I$.

Two applications follow: the Virasoro minimal models [BPZ1984], built as quotients of the universal Virasoro vertex algebra by singular vectors and compared with their known characters [RC1985]; and *presentations*, which write a subalgebra as a universal vertex algebra on its strong generators modulo an ideal of relations.

Everything is computed weight by weight, so the algebras of this notebook have conformal weights (from a conformal vector, or from `degrees=`, see `01_getting_started.ipynb`, sections 4 and 5). An important question for every result is whether it holds in *all* weights or only in the weights that were computed; section 2 explains how the library answers it.

**Prerequisites:** `01_getting_started.ipynb`.

**Elsewhere:** commutants and cosets, another kind of subalgebra, are in `07_commutants_and_cosets.ipynb`; Zhu algebras of quotients in `09_zhu_algebras.ipynb` (section 6); modules over quotients in `10_modules.ipynb` (section 8); ideals generated by singular vectors of affine vertex algebras in `13_singular_vectors_in_affine_vertex_algebras.ipynb`.

**Contents**

1. Subalgebras
2. Exact answers and answers up to a weight
3. Ideals
4. Quotients
5. Minimal models
6. Presentations
7. Limitations

## 1. Subalgebras

`V.subalgebra(A)` returns the vertex subalgebra $S$ generated by the homogeneous elements $A$, together with a set $G$ of strong generators of $S$. These are found by a *closure*:

1. start with $G = A$;
2. compute all products $g_{(n)}g'$, $n \ge 0$, of elements of $G$ (the poles of their OPEs);
3. a product that is not a linear combination of normally ordered monomials in $G$ becomes a new strong generator;
4. repeat with the enlarged $G$ until a round adds nothing.

When the closure stops, $G$ strongly generates $S$: every element of $S$ is a linear combination of normally ordered monomials in the elements of $G$ and their derivatives.

The first example is the Heisenberg algebra $H$, with one field $J$ and $J(z)J(w) \sim 1/(z-w)^2$, and its conformal vector $T = \frac12{:}JJ{:}$ of central charge 1. The cell below builds $H$ and the subalgebra $S$ generated by $T$, named `T` with `names=`. `S.strong_generator_info()` lists each strong generator with its conformal weight, its parity and where it came from (`'given'` for an element of $A$).

In [1]:
import vertex_algebras as va
from vertex_algebras import *

X.<J> = NormallyOrderedExpressions(QQ)
H = UniversalVertexAlgebra(QQ, X, {('J', 'J'): [0, 1]}, conformal_vector=J*J/2)
J = H.gen('J')
T = H.conformal_vector()
S = H.subalgebra([T], names=['T'])
S, S.strong_generator_info()

(Vertex subalgebra of Universal vertex algebra on generators ['J'] over Rational Field strongly generated by ['T'],
 [('T', 2, 0, 'given')])

The closure stops at once: $T$ alone strongly generates $S$, so $S$ is spanned by the normally ordered monomials in $T$ and its derivatives. The cell below compares the dimensions of the weight spaces of $H$ and of $S$ in the weights $0, \dots, 13$; $S$ is much smaller.

In [2]:
[H.weight_space(n).dimension() for n in range(14)], [S.weight_space(n).dimension() for n in range(14)]

([1, 1, 2, 3, 5, 7, 11, 15, 22, 30, 42, 56, 77, 101],
 [1, 0, 1, 1, 2, 2, 4, 4, 7, 8, 12, 14, 21, 24])

The universal Virasoro vertex algebra at $c = 1$ maps onto $S$ by $L \mapsto T$, and its weight spaces have as bases the monomials $L_{-n_1}\cdots L_{-n_r}\mathbf 1$ with $n_1 \ge \dots \ge n_r \ge 2$. Their dimensions are therefore the numbers of partitions of $n$ into parts $\ge 2$. The cell below checks that the dimensions of $S$ are these numbers, so the map is an isomorphism in the weights $\le 13$. (It is an isomorphism in every weight, because the universal Virasoro vertex algebra at $c = 1$ is simple [GK2007]; that is a theorem, not something the computation shows.)

In [3]:
[S.weight_space(n).dimension() for n in range(14)] == [Partitions(n, min_part=2).cardinality() for n in range(14)]

True

**Membership.** `S.contains(v)`, or `v in S`, decides exactly whether an element $v$ of $H$ lies in $S$. The cell below tests four elements: $J$ and ${:}JJJ{:}$ (both odd under $J \mapsto -J$, so not in $S$), ${:}TT{:}$, and ${:}J\,\partial J{:}$, which equals $\partial T$.

In [4]:
J in S, T.no(T) in S, J.no(J.derivative()) in S, J.no(J.no(J)) in S

(False, True, True, False)

**Expressions in the strong generators.** `S.express(v)` writes an element of $S$ as a linear combination of normally ordered monomials in the strong generators. The result is an element of `NormallyOrderedExpressions` (products there are nested to the right, as in `01_getting_started.ipynb`, section 2), and `S(e)` evaluates such an expression in $S$ again. The cell below expresses $x = {:}TT{:} + 2\partial^2 T$, which prints in the generator $J$ of $H$, and checks that evaluating the expression gives back $x$. When the monomials in the strong generators satisfy relations (section 6), `express` chooses one of the possible expressions; it is not canonical.

In [5]:
x = T.no(T) + 2*T.derivative(2)
e = S.express(x)
x, e, S(e) == x

(1/4*:J J J J: + 2*:dJ dJ: + 5/2*:d^2J J:, :T T: + 2*d^2T, True)

**Elements, lift and retract.** Elements of $S$ are elements of $H$ that lie in $S$; they print in the generators of $H$. Two maps go between them: `S.lift(s)` is $s$ viewed as an element of $H$, and `S.retract(v)` (or `S(v)`) is an element $v \in H$ viewed as an element of $S$. Elements of $S$ coerce into $H$, so a sum of an element of $S$ and an element of $H$ lies in $H$. The first cell below checks these four facts; the second shows the coercion, which is the embedding $S \to H$ as a morphism of vertex algebras (see `10_modules.ipynb`, section 9); the third shows that an element outside $S$ cannot be retracted.

In [6]:
Ts = S.gen('T')
Ts.parent() is S, S.lift(Ts).parent() is H, S.retract(J.no(J)), (Ts + J).parent() is H

(True, True, :J J:, True)

In [7]:
H.coerce_map_from(S)

Vertex algebra morphism:
  From: Vertex subalgebra of Universal vertex algebra on generators ['J'] over Rational Field strongly generated by ['T']
  To:   Universal vertex algebra on generators ['J'] over Rational Field
  Defn: T |--> 1/2*:J J:

In [8]:
S.retract(J)

ValueError: J is not an element of Vertex subalgebra of Universal vertex algebra on generators ['J'] over Rational Field strongly generated by ['T']

**Printing in the strong generators.** By default elements of $S$ print in the generators of $H$. `S.print_options(generators='strong')` prints them in the strong generators of $S$ instead. The cell below prints a basis of the weight-4 space of $S$ in both ways. The option belongs to $S$ (all its elements change), so it is set back afterwards.

In [9]:
S.print_options(generators='strong'); print(S.weight_space(4).basis())
S.print_options(generators='ambient'); print(S.weight_space(4).basis())

[d^2T, :T T:]
[:dJ dJ: + :d^2J J:, 1/4*:J J J J: + 1/2*:d^2J J:]


**A larger subalgebra.** To demonstrate that the closure operation can find new strong generators, take $T$ and ${:}JJJ{:}$ (named `W`): the OPE pole $T_{(3)}{:}JJJ{:} = 3J$ is not a combination of monomials in $T$ and ${:}JJJ{:}$, so the closure adds it as a new strong generator, named `S0` (the names of the generators found are `S0, S1, ...`). The cell below prints the strong generators with their origin. Since $J$ generates all of $H$, this subalgebra is $H$ itself.

In [10]:
S2 = H.subalgebra([T, J.no(J.no(J))], names=['T', 'W'])
S2.strong_generator_info(), S2.gen('S0')

([('T', 2, 0, 'given'), ('W', 3, 0, 'given'), ('S0', 1, 0, 'T_(3) W')], 3*J)

**The fixed points of $J \mapsto -J$.** The map $J \mapsto -J$ is an automorphism of $H$; its fixed points form the subalgebra $H^+$ spanned by the monomials with an even number of factors. Take $T$ and $W = {:}JJJJ{:}$: both lie in $H^+$, and the closure adds no generator, so $T$ and $W$ strongly generate a subalgebra $S_3 \subset H^+$. Dong and Nagatomo show that $H^+$ is generated by $T$ and a field of weight 4 [DN1999].

The character of $H^+$ is half the sum of the characters of $H$ and of the twist of $H$ by $J \mapsto -J$,
$$\operatorname{ch} H^+ = \frac12\Big(\prod_{n \ge 1}\frac{1}{1-q^n} + \prod_{n\ge1}\frac{1}{1+q^n}\Big).$$
The cell below compares the dimensions of $S_3$ with the coefficients of this series up to weight 15: they agree, so $S_3 = H^+$ in these weights.

In [11]:
S3 = H.subalgebra([T, J.no(J.no(J.no(J)))], names=['T', 'W'])
dims = [S3.weight_space(n).dimension() for n in range(16)]
PS.<q> = PowerSeriesRing(QQ, default_prec=16)
fixed = (1/prod(1 - q^n for n in range(1, 16)) + 1/prod(1 + q^n for n in range(1, 16)))/2
S3.variable_names(), dims, dims == fixed.padded_list(16)

(('T', 'W'), [1, 0, 1, 1, 3, 3, 6, 7, 12, 14, 22, 27, 40, 49, 69, 86], True)

## 2. Exact answers and answers up to a weight

A vertex algebra is infinite-dimensional, but a computation only sees finitely many weights. The library makes this explicit: each method says, through its name or its result, how far its answer goes.

* **Subalgebras are exact.** A subalgebra is only created once its closure has finished, so its strong generators are complete, and its weight spaces and membership tests are exact in every weight. If the closure does not finish within `max_closure_steps` rounds (by default 10), the constructor raises a `StrongGenerationError` and returns nothing, not a partial answer. With a single round, the closure for $T$ and ${:}JJJ{:}$ of section 1 finds `S0` but cannot confirm that no further generator is needed:

In [12]:
H.subalgebra([T, J.no(J.no(J))], names=['T', 'W3'], max_closure_steps=1)

StrongGenerationError: no strong generating set found within 1 closure round(s) (the subalgebra may not be finitely strongly generated); strong generators so far: T (weight 2), W3 (weight 3), S0 (weight 1); 3 pair(s) of generators untested; use a larger max_closure_steps

* **Ideals and quotients are exact weight by weight** (sections 3 and 4). Each weight space is computed exactly, but a list of dimensions, or a character `Q.character(N)`, says nothing about the weights $\ge N$.

Other constructions of the library are only known up to a weight, and their methods carry the limit in their names: commutants in `07_commutants_and_cosets.ipynb` (section 1) and Zhu algebras in `09_zhu_algebras.ipynb`.

## 3. Ideals

`V.ideal(gens)` is the ideal generated by the elements `gens`: the smallest subspace of $V$ that contains them and is stable under $\partial$ and under all products $a_{(n)}$, $a \in V$, $n \in \mathbb Z$. An ideal is a graded subspace of $V$, not a vertex algebra. Each of its weight spaces is computed exactly, from the modes of the strong generators of $V$ applied to the generators of the ideal; the normal forms below are those of the Gröbner–Shirshov basis method for vertex algebras [KK2023].

The example is the universal affine vertex algebra $V^1(\mathfrak{sl}_2)$ of level 1 (see `03_affine_vertex_algebras.ipynb`), with generators named $E, H, F$ and the Sugawara conformal vector. The ideal $I$ is generated by ${:}EE{:}$ (with the generator of the ideal named `EE`). The cell below prints the ideal and the dimensions of the weight spaces of $V^1(\mathfrak{sl}_2)$ and $I$ up to weight 6.

In [13]:
A = va.Affine(QQ, ['A', 1], 1, names=['E', 'H', 'F'], conformal_vector='sugawara')
E, Hc, F = A.gens()
I = A.ideal([E.no(E)], names=['EE'])
I, [A.weight_space(n).dimension() for n in range(7)], [I.weight_space(n).dimension() for n in range(7)]

(Ideal of Universal affine vertex algebra of type ['A', 1] at level 1 over Rational Field generated by [:E E:],
 [1, 3, 9, 22, 51, 108, 221],
 [0, 0, 5, 15, 38, 89, 192])

The weight-2 space of $I$ is 5-dimensional. The element ${:}EE{:}$ has $\mathfrak{sl}_2$-weight ($h_{0}$-eigenvalue) 4 and is killed by $E_{(0)}$, so the zero modes of the currents acting on it generate the 5-dimensional representation of $\mathfrak{sl}_2$, which is all of the degree 2 subspace $I_2$.

The basis of a weight space of an ideal is in reduced echelon form. Its pivots, the *leading monomials* of the ideal, are the basis monomials of $V$ that the ideal eliminates in the quotient (section 4); `I.weight_space(n).leading_monomials()` returns them as basis keys. The cell below prints the basis of $I_2$.

In [14]:
B = I.weight_space(2)
B.basis()

[:F F:, :H F: + dF, -2*:E F: + :H H: + dH, :E H: + dE, :E E:]

**Membership and certificates.** `I.contains(v)`, or `v in I`, is exact. The first cell below tests three elements of weight 2: ${:}FF{:}$, ${:}EH{:} + \partial E$ (both in $I_2$ above) and ${:}EF{:}$ (not in $I$).

`I.express(v)` writes an element of the ideal as an explicit linear combination of modes of $V$ applied to the generators of the ideal (here the one generator `EE`). This is a *certificate* that $v$ lies in $I$, which can be checked independently by evaluating it with `.element()`. The second cell finds ${:}FF{:} = \frac1{24}F_{(0)}^4\,{:}EE{:}$ and checks it.

In [15]:
F.no(F) in I, E.no(Hc) + E.derivative() in I, E.no(F) in I

(True, True, False)

In [16]:
c1 = I.express(F.no(F))
c1, c1.element() == F.no(F)

(1/24*F_(0) F_(0) F_(0) F_(0) EE, True)

**Normal forms.** `I.normal_form(v)` subtracts from $v$ an element of the ideal so that no leading monomial of the ideal remains. Two elements have the same normal form exactly when their difference lies in $I$, and the normal form of $v$ is zero exactly when $v \in I$. For example the normal form of ${:}EH{:}$ is $-\partial E$, because ${:}EH{:} + \partial E \in I$:

In [17]:
I.normal_form(E.no(Hc)), I.normal_form(E.no(F)), I.normal_form(F.no(F))

(-dE, :E F:, 0)

## 4. Quotients

`V.quotient(I)`, or `V / I`, is the quotient vertex algebra $V/I$; `I` can be an ideal or a list of its generators. An element of $Q = V/I$ is stored as its normal form (section 3), so it prints as a combination of the *standard monomials*: the basis monomials of $V$ that are not leading monomials of $I$. The main conversions are:

* `Q(v)` is the image of $v \in V$ in $Q$, and elements of $V$ coerce into $Q$;
* `Q.lift(x)` is the canonical representative of $x \in Q$ in $V$, its normal form;
* products of elements of $Q$ are computed in $V$ and reduced to normal form.

Quotients have unique representation, so `A / I` and `A.quotient(I)` give the same object. As everywhere in the library, the quotient has no conformal vector unless one is asked for:

In [18]:
Q = A / I
Q is A.quotient(I), Q.conformal_vector()

(True, None)

`conformal_vector='ambient'` gives $Q$ the image of the conformal vector of $V$, here the Sugawara vector of central charge $3k/(k+2) = 1$:

In [19]:
Q = A.quotient(I, conformal_vector='ambient')
Q.central_charge()

1

The generators of $Q$ are the images of $E, H, F$. In $Q$ the relations of section 3 hold: ${:}EE{:} = 0$, ${:}EH{:} = -\partial E$ and ${:}FF{:} = 0$, while the OPE of $E$ and $F$ is unchanged. The second cell shows an element as an element of $Q$ and its lift to $V$.

In [20]:
Eq, Hq, Fq = Q.gens()
Eq.no(Eq), Eq.no(Hq), Fq.no(Fq), Eq.ope(Fq)

(0, -dE, 0, [H, 1])

In [21]:
x = Q(E.no(Hc))
x, x.parent() is Q, Q.lift(x), Q.lift(x).parent() is A

(-dE, True, -dE, True)

The weight spaces of $Q$ have the standard monomials as bases. In weight 2, the five leading monomials of $I_2$ are missing:

In [22]:
Q.weight_space(2).basis()

[:E F:, dE, dH, dF]

**Comparison with a lattice vertex algebra.** By the Frenkel–Kac construction [FK1980], the simple affine vertex algebra $L_1(\mathfrak{sl}_2)$ is the lattice vertex algebra $V_{A_1}$ of the root lattice $\mathbb Z\alpha$, $(\alpha|\alpha) = 2$, with the conformal vector $\frac14{:}JJ{:}$ (see `05_lattice_vertex_algebras.ipynb`, sections 1 and 2). Its character is
$$\operatorname{ch} V_{A_1} = \frac{\sum_{n \in \mathbb Z} q^{n^2}}{\prod_{m \ge 1}(1-q^m)}.$$
The cell below shows that $Q$ has this character up to $q^9$. This compares dimensions only, and only up to weight 9; that $Q$ is $L_1(\mathfrak{sl}_2)$, i.e. that ${:}EE{:}$ generates the maximal ideal of $V^1(\mathfrak{sl}_2)$, is a theorem [Kac1998]. (The morphism $V^1(\mathfrak{sl}_2) \to V_{A_1}$ itself is built in `10_modules.ipynb`, section 9.)

In [23]:
LA = va.Lattice(QQ, matrix([[2]]))
LA = LA.with_conformal_vector(LA.J(0).no(LA.J(0))/4)
Q.character(10), Q.character(10) == LA.character(10)

(1 + 3*q + 4*q^2 + 7*q^3 + 13*q^4 + 19*q^5 + 29*q^6 + 43*q^7 + 62*q^8 + 90*q^9 + O(q^10),
 True)

## 5. Minimal models

For coprime integers $p, p' \ge 2$ let
$$c_{p,p'} = 1 - \frac{6(p-p')^2}{pp'}.$$
The Virasoro minimal model of central charge $c_{p,p'}$ [BPZ1984] has as its vertex algebra the simple quotient $L(c_{p,p'}, 0)$ of the universal Virasoro vertex algebra $V^c$ at $c = c_{p,p'}$. Its character, found by Rocha-Caridi [RC1985], is
$$\chi_{p,p'}(q) = \frac{\sum_{k \in \mathbb Z} \bigl(q^{k(kpp' + p - p')} - q^{(kp+1)(kp'+1)}\bigr)}{\prod_{n \ge 1}(1-q^n)}.$$
This section finds the singular vectors of $V^c$ with $c$ as a parameter, builds the quotients by them, and compares the quotients with $\chi_{p,p'}$.

**Where singular vectors exist.** A *singular vector* of weight $N$ is a nonzero $v \in V^c_N$ with $L_n v = 0$ for all $n \ge 1$. Since $L_1$ and $L_2$ generate all $L_n$, $n \ge 1$, it suffices to show that $L_1 v = L_2 v = 0$. The function `singular_matrix` below writes the linear map $v \mapsto (L_1 v, L_2 v)$ on $V^c_N$ as a matrix over $\mathbb Q[c]$ (the basis of $V^c_N$ does not depend on $c$). For generic $c$ its columns are linearly independent, and they become dependent exactly at the roots of the gcd $D_N(c)$ of its maximal minors. So the roots of $D_N$ are exactly the central charges at which $V^c$ has a singular vector of weight $N$. The cell below computes $D_N$ for $N = 4, 6, 8$ and factors it.

In [24]:
R.<c> = QQ['c']
Vc = va.Virasoro(R, c, conformal_vector='L')

def singular_matrix(V, N):
    # the matrix of v |-> (L_1 v, L_2 v) on the weight space V_N
    L = V.gen('L')
    basis = V.weight_space(N).basis()
    rows = []
    for n in (1, 2):
        images = [L.mode(v, n) for v in basis]
        rows += [[im.coefficient(k) for im in images] for k in V.weight_space(N - n).basis_keys()]
    return matrix(V.base_ring(), rows)

D = {N: gcd(singular_matrix(Vc, N).minors(Vc.weight_space(N).dimension())) for N in (4, 6, 8)}
{N: d.factor() for N, d in D.items()}

{4: c + 22/5, 6: (c - 1/2) * (c + 68/7), 8: (c + 3/5) * (c + 46/3)}

Each root is a minimal-model central charge $c_{p,p'}$, with $(p-1)(p'-1) = N$:

| $N$ | roots | $(p, p')$ |
|---|---|---|
| 4 | $-22/5$ | $(2, 5)$, the Lee–Yang model |
| 6 | $1/2$, $-68/7$ | $(3, 4)$, the Ising model; $(2, 7)$ |
| 8 | $-3/5$, $-46/3$ | $(3, 5)$; $(2, 9)$ |

The function `minimal_pair` in the cell below finds $(p, p')$ for each root. At a fixed central charge, `V.find_singular(N)` returns a basis of the singular vectors of weight $N$ (the vectors killed by the positive modes of the strong generators). The cell prints the list of cases, and the singular vectors of the Lee–Yang model, $\Lambda = {:}LL{:} - \frac3{10}\partial^2 L$ of weight 4, and of the Ising model, of weight 6.

In [25]:
def minimal_pair(c0, N):
    return next((p, pp) for p in range(2, N + 2) for pp in range(p + 1, N + 2)
                if gcd(p, pp) == 1 and (p - 1)*(pp - 1) == N and 1 - 6*(p - pp)^2/(p*pp) == c0)

cases = [(N, c0, minimal_pair(c0, N)) for N, d in D.items() for c0, _ in d.roots()]
LeeYang = va.Virasoro(QQ, -22/5, conformal_vector='L')
Ising = va.Virasoro(QQ, 1/2, conformal_vector='L')
cases, LeeYang.find_singular(4), Ising.find_singular(6)

([(4, -22/5, (2, 5)),
  (6, 1/2, (3, 4)),
  (6, -68/7, (2, 7)),
  (8, -3/5, (3, 5)),
  (8, -46/3, (2, 9))],
 [:L L: - 3/10*d^2L],
 [:L L L: + 93/64*:dL dL: - 33/16*:d^2L L: - 9/128*d^4L])

**The quotients.** For each case the cell below builds the quotient $Q = V^{c_{p,p'}}/\langle v\rangle$, for $v$ the singular vector of weight $N$, and compares the dimensions of its weight spaces with the coefficients of $\chi_{p,p'}$ (computed by the function `vacuum_character`) up to weight 20. It also prints, for the Lee–Yang quotient, the relation ${:}LL{:} = \frac3{10}\partial^2L$ and the character up to $q^{11}$.

In [26]:
def vacuum_character(p, pp, prec):
    # Rocha-Caridi's vacuum character of M(p, p'), up to q^(prec-1)
    PS.<q> = PowerSeriesRing(ZZ, default_prec=prec)
    numer = PS(0)
    for k in range(-prec, prec + 1):
        for e, sign in ((k*(k*p*pp + p - pp), 1), ((k*p + 1)*(k*pp + 1), -1)):
            if 0 <= e < prec:
                numer += sign*q^e
    return numer / prod(1 - q^n for n in range(1, prec))

results = []
for N, c0, (p, pp) in cases:
    V0 = va.Virasoro(QQ, c0, conformal_vector='L')
    Q0 = V0.quotient(V0.find_singular(N), conformal_vector='ambient')
    dims = [Q0.weight_space(n).dimension() for n in range(21)]
    results.append(((p, pp), c0, dims == vacuum_character(p, pp, 21).padded_list(21)))
    if (p, pp) == (2, 5):
        LY = Q0
Lq = LY.gen('L')
results, Lq.no(Lq), LY.character(12)

([((2, 5), -22/5, True),
  ((3, 4), 1/2, True),
  ((2, 7), -68/7, True),
  ((3, 5), -3/5, True),
  ((2, 9), -46/3, True)],
 3/10*d^2L,
 1 + q^2 + q^3 + q^4 + q^5 + 2*q^6 + 2*q^7 + 3*q^8 + 3*q^9 + 4*q^10 + 4*q^11 + O(q^12))

In all five cases the quotient by the singular vector of weight $(p-1)(p'-1)$ has the dimensions of $L(c_{p,p'}, 0)$ up to weight 20, and the computation shows no more than that. The fact that the quotient is simple is a theorem of Wang [Wan1993]: at $c = c_{p,p'}$ this singular vector generates the maximal ideal of $V^c$.

## 6. Presentations

A subalgebra $S$ with strong generators $s_1, \dots, s_r$ can be written as a quotient $F/R$:

* $F$ is the universal vertex algebra [DSK2005] on generators $s_1, \dots, s_r$, whose OPEs are those of $S$, written in normally ordered monomials of the $s_i$ (using `S.express`);
* $R$ is the kernel of the map $F \to S$ that sends each normally ordered monomial to the same monomial in $S$. This map respects all products, so $R$ is an ideal and $F/R \cong S$.

The OPEs need not satisfy the Jacobi identity in $F$: the expressions chosen for the poles are only correct modulo $R$. Then $F$ is not a vertex algebra. The library calls such an object a *proto vertex algebra* (the category `ProtoVertexAlgebras`): it has a vacuum, a translation $\partial$ and $n$-th products for all $n \in \mathbb Z$, satisfying the vacuum and translation axioms, but the Jacobi identity is not claimed. This is the library's own term, not a standard notion in the literature. The failures of the Jacobi identity lie in $R$, and `F.jacobi_identity_failures()` lists them (see `12_ope_bootstrap.ipynb`).

`S.presentation()` returns $F/R$, as a quotient of $F$. `P.ambient()` is $F$, and `P.relations(w)` is a basis of the weight-$w$ space $R_w$, computed exactly as the kernel of $F_w \to S_w$. Elements convert both ways: `P(s)` for $s \in S$, and `S(x)` for $x \in P$.

**The Ising model inside a free fermion.** The Virasoro field $T = \frac12{:}\partial\psi\,\psi{:}$ of a Majorana fermion $\psi(z)\psi(w) \sim 1/(z-w)$ (`va.FreeFermions`, see `04_ghost_systems.ipynb`, section 5) generates a Virasoro vertex algebra at $c = \frac12$. The cell below builds the subalgebra $S$ generated by $T$ and its presentation. $F$ is the universal Virasoro algebra $V^{1/2}$ (its OPE has the fourth-order pole $c/2 = 1/4$), and $R$ is zero below weight 6 and one-dimensional in weight 6.

In [27]:
Fer = va.FreeFermions(QQ)
psi = Fer.gen(0)
SI = Fer.subalgebra([psi.derivative().no(psi)/2], names=['L'])
P = SI.presentation()
FL = P.ambient()
FL, FL.gen('L').ope(FL.gen('L')), [len(P.relations(w)) for w in range(7)]

(Universal vertex algebra on generators ['L'] over Rational Field,
 [dL, 2*L, 0, 1/4],
 [0, 0, 0, 0, 0, 0, 1])

The relation of weight 6 is the Ising singular vector found by `find_singular` in section 5:

In [28]:
r6 = P.relations(6)[0]
r6, repr(r6) == repr(Ising.find_singular(6)[0])

(:L L L: + 93/64*:dL dL: - 33/16*:d^2L L: - 9/128*d^4L, True)

The dimensions of $S$ agree with Rocha-Caridi's character $\chi_{3,4}$ of $L(\frac12, 0)$ up to weight 20 (cell below). Section 5 showed the same for the quotient of $V^{1/2}$ by the ideal generated by this singular vector. That ideal lies in $R$, so up to weight 20 it is all of $R$: in these weights $S$ agrees with the Ising quotient of section 5. (Again, the statement in all weights is the theorem of [Wan1993], together with the simplicity of $S$.)

In [29]:
[SI.weight_space(w).dimension() for w in range(21)] == vacuum_character(3, 4, 21).padded_list(21)

True

**Conversions between $S$ and $P$.** The normally ordered cube ${:}LLL{:}$ is not a standard monomial of $P$ (it is the leading monomial of the relation of weight 6), so it prints as its normal form. The cell below converts it to $S$, where it is the cube of the generator of $S$, and converts the generator of $S$ to $P$.

In [30]:
L = P.gen('L')
x = L.no(L.no(L))
Ls = SI.gen('L')
x, SI(x) == Ls.no(Ls.no(Ls)), P(Ls) == L

(-93/64*:dL dL: + 33/16*:d^2L L: + 9/128*d^4L, True, True)

**A presentation with a non-trivial Jacobi check.** For the subalgebra $S_3 = H^+$ of section 1, strongly generated by $T$ and $W = {:}JJJJ{:}$ of weights 2 and 4, the cell below shows that the OPEs satisfy the Jacobi identity in $F$ (no failures), and compares the dimensions of $F$ and $S_3$ with the number of relations in each weight. They agree below weight 8, and the first relation is in weight 8; in every weight, $\dim F_w - \dim (S_3)_w$ is the number of relations.

In [31]:
P3 = S3.presentation()
F3 = P3.ambient()
(F3.jacobi_identity_failures(), [F3.weight_space(n).dimension() for n in range(12)],
 [S3.weight_space(n).dimension() for n in range(12)], [len(P3.relations(w)) for w in range(12)])

({},
 [1, 0, 1, 1, 3, 3, 6, 7, 13, 16, 26, 33],
 [1, 0, 1, 1, 3, 3, 6, 7, 12, 14, 22, 27],
 [0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 4, 6])

## 7. Limitations

* **Generators of subalgebras.** `V.subalgebra(A)` needs homogeneous elements of positive conformal weight, or a grading that makes some twisted weight positive (`horizontal_grading='ambient'`, used for the triplet algebras of `14_triplet_algebra.ipynb` and `15_triplet_modules.ipynb`). If the closure does not finish within `max_closure_steps` rounds, there is no partial result; a subalgebra that is not finitely strongly generated cannot be constructed.
* **Cost of the closure.** Every round computes all poles $g_{(n)}g'$ of the strong generators and tests them against the spans of normally ordered monomials up to weight $\Delta_g + \Delta_{g'} - 1$; for generators of large weight this is the expensive step (see `14_triplet_algebra.ipynb`).
* **Ideals and quotients** need conformal weights on $V$ with finite-dimensional weight spaces (or a horizontal grading with finite-dimensional graded pieces). Their weight spaces are computed one at a time, so statements about all weights (that an ideal is maximal, that a quotient is simple, a character in closed form) are theorems to cite, not outputs: compare the uses of [GK2007], [Wan1993] and [Kac1998] above.
* **Expressions are not canonical.** `S.express(v)` returns one expression when the monomials in the strong generators are linearly dependent; `I.express(v)` returns one certificate among many.
* **Singular vectors** are found by `find_singular` at a fixed central charge (or level). A search over a parameter, as in section 5, is done by hand with linear algebra over a polynomial ring.
* **Presentations.** `P.relations(w)` is a basis of the relations in weight $w$. The library does not determine a finite set of generators of the ideal $R$, so a presentation is exact weight by weight but is not a finite presentation.

## References

- [BPZ1984] A. A. Belavin, A. M. Polyakov, A. B. Zamolodchikov, *Infinite conformal symmetry in two-dimensional quantum field theory*, Nuclear Phys. B 241 (1984), 333–380, [doi:10.1016/0550-3213(84)90052-X](https://doi.org/10.1016/0550-3213(84)90052-X). — the minimal models (section 5).
- [DN1999] C. Dong, K. Nagatomo, *Classification of irreducible modules for the vertex operator algebra $M(1)^+$*, J. Algebra 216 (1999), 384–404, [arXiv:math/9806051](https://arxiv.org/abs/math/9806051). — the fixed points of the Heisenberg algebra under $J \mapsto -J$ (section 1).
- [DSK2005] A. De Sole, V. G. Kac, *Freely generated vertex algebras and non-linear Lie conformal algebras*, Comm. Math. Phys. 254 (2005), 659–694, [arXiv:math-ph/0312042](https://arxiv.org/abs/math-ph/0312042). — strong generators, universal vertex algebras (sections 1, 6).
- [FK1980] I. B. Frenkel, V. G. Kac, *Basic representations of affine Lie algebras and dual resonance models*, Invent. Math. 62 (1980), 23–66, [doi:10.1007/BF01391662](https://doi.org/10.1007/BF01391662). — $L_1(\mathfrak{sl}_2)$ as a lattice vertex algebra (section 4).
- [GK2007] M. Gorelik, V. G. Kac, *On simplicity of vacuum modules*, Adv. Math. 211 (2007), 621–677, [arXiv:math-ph/0606002](https://arxiv.org/abs/math-ph/0606002). — the universal Virasoro vertex algebra is simple for $c \neq c_{p,p'}$ (section 1).
- [Kac1998] V. G. Kac, *Vertex algebras for beginners*, 2nd ed., Univ. Lecture Ser. 10, AMS (1998), [doi:10.1090/ulect/010](https://doi.org/10.1090/ulect/010). — strong generators, ideals, the Frenkel–Kac construction (sections 1, 3, 4).
- [KK2023] R. A. Kozlov, P. S. Kolesnikov, *The Gröbner–Shirshov bases method for vertex algebras*, J. Math. Sci. 296 (2026), 217–230, [arXiv:2312.01670](https://arxiv.org/abs/2312.01670). — ideals and normal forms (sections 3, 4).
- [RC1985] A. Rocha-Caridi, *Vacuum vector representations of the Virasoro algebra*, in: Vertex operators in mathematics and physics, Math. Sci. Res. Inst. Publ. 3, Springer (1985), 451–473, [doi:10.1007/978-1-4613-9550-8_22](https://doi.org/10.1007/978-1-4613-9550-8_22). — the vacuum characters of the minimal models (sections 5, 6).
- [Wan1993] W. Wang, *Rationality of Virasoro vertex operator algebras*, Int. Math. Res. Not. 1993, no. 7, 197–211, [doi:10.1155/S1073792893000212](https://doi.org/10.1155/S1073792893000212). — the singular vector generates the maximal ideal (sections 5, 6).